In [ ]:
import os, json, time
import pandas as pd
import requests
from tqdm import tqdm
from dotenv import load_dotenv
from helpers import get_sh_token, request_with_retry

load_dotenv()

# Notebook: Retrieve Sentinel-2 Scene Classification Layer (SCL) Statistics

This notebook retrieves the Scene Classification Layer (SCL) statistics for each training window using the Copernicus Data Space Ecosystem Statistics API. The SCL provides information about the type of surface (e.g., vegetation, water, clouds) for each pixel in a Sentinel-2 image.

Purpose: Enhance the training dataset by including SCL statistics for each window, used later for splitting the dataset into training, validation, and test sets.

In [ ]:
IN_PARQUET = "parquet/windows_best_s2_with_vza_angleDiffLE4.parquet"
OUT_PARQUET = "parquet/windows_best_s2_with_vza_angleDiffLE4_withSCL.parquet"

CLIENT_ID = os.getenv("CLIENT_ID")
CLIENT_SECRET = os.getenv("CLIENT_SECRET")

RES_METERS = 20 # SCL is natively 20m resolution

## Evalscript Definition

Define the Sentinel Hub evalscript to extract SCL.

In [ ]:
# --- Evalscript: output the SCL class as a numeric band ---
EVALSCRIPT_SCL = """//VERSION=3
function setup() {
  return {
    input: [{ bands: ["SCL", "dataMask"] }],
    output: [
      { id: "scl", bands: 1, sampleType: "UINT8" },
      { id: "dataMask", bands: 1 }
    ]
  };
}

function evaluatePixel(s) {
  return {
    scl: [s.SCL],
    dataMask: [s.dataMask]   // IMPORTANT for Stats API
  };
}
"""

In [ ]:
def build_payload(bbox, day_from, day_to):
    return {
        "input": {
            "bounds": {
                "bbox": bbox,
                "properties": {"crs": "http://www.opengis.net/def/crs/EPSG/0/3035"},
            },
            "data": [{
                "type": "sentinel-2-l2a",
                "dataFilter": {"mosaickingOrder": "mostRecent"},
            }],
        },
        "aggregation": {
            "timeRange": {"from": day_from, "to": day_to},
            "aggregationInterval": {"of": "P1D"},
            "evalscript": EVALSCRIPT_SCL,
            "resx": RES_METERS,
            "resy": RES_METERS,
        },
        "calculations": {
            "default": {
                "histograms": {
                    "default": {
                        # include classes 0..11 => bins 0..11
                        "binWidth": 1,
                        "lowEdge": 0,
                        "highEdge": 12
                    }
                }
            }
        }
    }

def parse_scl_counts(js):
    b0 = js["data"][0]["outputs"]["scl"]["bands"]["B0"]
    counts = {int(b["lowEdge"]): int(b["count"]) for b in b0["histogram"]["bins"]}
    return counts

def normalize_scl_counts(x):
    if x is None or (isinstance(x, float) and pd.isna(x)):
        return None
    if isinstance(x, dict):
        return {str(k): int(v) for k, v in x.items()}
    # if something weird slipped in, store as string so parquet can write it
    return {"_raw": str(x)}

In [ ]:
df = pd.read_parquet(IN_PARQUET)

token = get_sh_token()
scl_dicts = [None] * len(df)
errors = [None] * len(df)

In [ ]:
ok = 0
err = 0

rows = df.itertuples(index=True)  # yields Index + columns
pbar = tqdm(rows, total=len(df), desc="SCL per window")

for row in pbar:
    i = row.Index  # dataframe index position
    bbox = [float(row.minx), float(row.miny), float(row.maxx), float(row.maxy)]

    t = pd.to_datetime(row.s2_time, utc=True)
    day_start = t.normalize()
    day_end = day_start + pd.Timedelta(days=1)
    day_from = day_start.strftime("%Y-%m-%dT%H:%M:%SZ")
    day_to   = day_end.strftime("%Y-%m-%dT%H:%M:%SZ")

    payload = build_payload(bbox, day_from, day_to)

    resp = request_with_retry(payload, token)

    if resp.status_code in (401, 403):
        token = get_sh_token()
        resp = request_with_retry(payload, token)

    if resp.status_code != 200:
        err += 1
        errors[i] = {"status": int(resp.status_code), "message": resp.text[:800]}
        pbar.set_postfix(ok=ok, err=err)
        continue

    js = resp.json()
    scl_dicts[i] = parse_scl_counts(js)
    ok += 1

    pbar.set_postfix(ok=ok, err=err)
    time.sleep(0.1)  # be nice to the API

df["scl_counts"] = scl_dicts
df["scl_error"] = errors

df["scl_counts_json"] = df["scl_counts"].apply(lambda d: json.dumps(d) if isinstance(d, dict) else None)
df.drop(columns=["scl_counts"], inplace=True)
df.to_parquet(OUT_PARQUET, index=False)

print("Saved:", OUT_PARQUET)
print(f"Done. OK: {ok} / {len(df)} | Errors: {err}")